# Lesson 29 — Reformulations and Symmetry

## Learning objectives

1. Recognize symmetric MILPs and the cost they impose on B&B.
2. Apply orbital, lex, and isomorphism-pruning techniques.
3. Recognize convexification reformulations: extended formulations, perspective.
4. Use `discopt`'s reformulation infrastructure / `/reformulate` skill.

## 1. Symmetry hurts B&B

If two variables $x_i, x_j$ are interchangeable (the constraint matrix is invariant under their swap), B&B explores symmetric subtrees independently. Each leaves equivalent solutions multiple times in the tree {cite:p}`Margot2010`.

Examples: bin packing with identical bins; multi-machine scheduling with identical machines; coloring with interchangeable colors.

## 2. Symmetry breaking

- **Lex constraints:** $x_1 \preceq_{\text{lex}} x_2 \preceq \dots$ — valid **only when the variables lie in a single orbit of the formulation's symmetry group**, i.e., they really are interchangeable. Adding lex on non-symmetric variables can cut off optimal solutions.
- **Orbital fixing/branching** {cite:p}`Margot2010`: at each node, identify orbits of the symmetry group; fix all members of an orbit consistently.
- **Isomorphism pruning:** in heavily symmetric problems (e.g., graph coloring), prune nodes that are isomorphic to one already explored.

Modern solvers detect simple symmetries automatically; complex ones require user input.

## 3. Reformulation as a reasoning task

Three families of reformulations matter most:

- **Extended formulations:** add auxiliary variables to make the formulation tighter (smaller LP-relaxation gap). Classic example: SOS2 piecewise via $\lambda$-binaries vs SOS-native.
- **Perspective reformulation:** for an *indicator-conditional* convex epigraph constraint of the form "$y = 1 \Rightarrow f(x) \le t$ and $y = 0 \Rightarrow x = 0$" with $y \in \{0, 1\}$ and $f$ convex, the convex hull of the disjunction is the **perspective** $y f(x / y) \le t$ (with the convention $0 \cdot f(0/0) = 0$). This is strictly tighter than the corresponding big-M and is a workhorse for fixed-cost / indicator constraints in MINLP {cite:p}`Trespalacios2015,Grossmann2013`.
- **Convexification of bilinear terms:** McCormick (Lesson 16) is one option; if the bilinear term has special structure (e.g., $x_i x_j$ both binary), there may be a stronger formulation.

{cite:p}`Liberti2009` is a systematic catalogue.

## 4. discopt's reformulation pipeline

`discopt.reformulate` runs:

1. **Detect** symmetries via constraint-matrix automorphisms.
2. **Detect** weak big-M and replace with tight bounds.
3. **Detect** disjunctive structure and apply convex-hull reformulation when better.
4. **Detect** bilinear terms and apply McCormick or RLT.

Logs explain each transformation. Use the `/reformulate` Claude Code skill (in `.claude/commands/reformulate.md`) to get an LLM-assisted suggestion list — but always verify.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

# A small symmetric MILP: assign 4 jobs to 3 identical machines, minimise makespan.
def build(symbreak: bool):
    m = do.Model("makespan_sb" if symbreak else "makespan")
    n_jobs, n_mach = 4, 3
    p = np.array([3, 5, 2, 4])
    x = m.binary("x", shape=(n_jobs, n_mach))
    C = m.continuous("C", lb=0, ub=20)
    for j in range(n_jobs):
        m.subject_to(dm.sum(x[j, :]) == 1)
    for k in range(n_mach):
        m.subject_to(p @ x[:, k] <= C)
    if symbreak:
        # Bin-load lex: heaviest machine first.
        loads = [p @ x[:, k] for k in range(n_mach)]
        for k in range(n_mach - 1):
            m.subject_to(loads[k] >= loads[k + 1])
    m.minimize(C)
    return m, x, C

for label, sb in [("no symmetry breaking", False), ("with bin-load lex", True)]:
    mm, _, _ = build(sb)
    r = mm.solve()
    print(f"{label:24s} nodes={r.node_count:4d}  time={r.wall_time:.3f}s  C*={r.objective}")


## 5. Reformulation as a research practice

When stuck on a slow MILP/MINLP:

1. Look at the LP relaxation gap. If big, formulation is weak.
2. Inspect big-M values; tighten via OBBT.
3. Check for symmetry; add lex constraints.
4. Look for disjunctive structure; consider convex-hull reformulation.
5. Look for bilinear/concave terms; apply tighter relaxation.

This loop is the difference between a 2-day solve and a 2-second solve.

## References
{cite:p}`Margot2010,Liberti2009,Williams2013`.